# files

> Files and cells over the gateway's contents and cells APIs

In [ ]:
#| default_exp files

[Rustygate](https://github.com/AnswerDotAI/rustygate) provides a Jupyter-style `/api/contents` API for files and directories and `/api/cells` for editing cells inside notebooks.

Use `JupyAsyncFilesClient` to address files by path. `JupyAsyncCellsClient` works with one notebook's cells. A kernel client bound to that notebook receives `cell_ops` change messages on its `cells` channel. `apply_ops` updates a local list of cells from those messages. Attach `JmsgQueues` if you want to read the broadcasts with `get_jmsg`.

In [ ]:
#| export
import json
from base64 import b64encode, b64decode
from fastcore.basics import patch, patch_to
from fastcore.meta import delegates
from fasttransport.errors import APIError
from jupyasyncclient.core import KernelApi

In [ ]:
import asyncio, tempfile
from pathlib import Path
from queue import Empty
from fastcore.test import test_eq, expect_fail
from rustygate.tools import start_gateway
from jupyasyncclient import JupyAsyncKernelClient, JmsgQueues

## The files client

Both clients inherit `KernelApi`'s base URL, authentication, and HTTP transport. You can supply an HTTP client or let the transport create one for each request. Writes replace the addressed content; the last accepted edit wins.

In [ ]:
#| export
class JupyAsyncFilesClient(KernelApi):
    "Files and directories over the gateway's contents API."

`_op` calls an operation from the bundled OpenAPI specification, omitting arguments with value `None`. HTTP errors propagate as `APIError`.

The generated operation separates query parameters from body fields. For example, `get(path)` requests a model and `put(path, type='directory')` requests directory creation. `patch` passes its body through `body_` because a rename has two paths: the URL's source path and the body's destination `path`.


In [ ]:
#| export
@patch
async def _op(self:JupyAsyncFilesClient, op, **kw):
    "Call spec op `op` with None values dropped."
    return await op(**{k:v for k,v in kw.items() if v is not None})


Use `get` to read a contents model, `put` to write, and `post` to copy through the contents API:

In [ ]:
#| export
@patch
async def get(self:JupyAsyncFilesClient, path='', **kwargs):
    "The model at `path`; `kwargs` become query parameters, e.g. `fields`."
    if not path: return await self._op(self.api.contents.get_root, **kwargs)
    return await self._op(self.api.contents.get_path, path=path, **kwargs)

@patch
async def put(self:JupyAsyncFilesClient, path, **kwargs):
    "PUT to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.put_path, path=path, **kwargs)

@patch
async def post(self:JupyAsyncFilesClient, path, **kwargs):
    "POST to the contents API; `kwargs` are its body and query fields."
    return await self._op(self.api.contents.post_path, path=path, **kwargs)

For `patch`, pass the source path positionally. You can then use `path=` for the destination in the body. `delete` removes a file or an empty directory:

In [ ]:
#| export
@patch_to(JupyAsyncFilesClient)
async def patch(self, path, /, **kwargs):
    "PATCH with `kwargs` as the JSON body; `path` is positional-only, freeing the name for the body."
    return await self._op(self.api.contents.patch_path, path=path, body_=kwargs)

@patch
async def delete(self:JupyAsyncFilesClient, path):
    "Delete a file or an empty directory."
    return await self._op(self.api.contents.delete_path, path=path)

In [ ]:
#| export
@patch
async def write(self:JupyAsyncFilesClient, path, content, unique=False, overwrite=True):
    "Write text or base64-encoded bytes and return the model. `unique` chooses a free `name_n.ext`; `overwrite=False` requires a new file."
    c,f = (b64encode(content).decode(),'base64') if isinstance(content, bytes) else (content,'text')
    return await self.put(path, overwrite=overwrite, unique=unique or None, content=c, format=f)

@patch
async def read(self:JupyAsyncFilesClient, path):
    "A file's contents: `str` for text, `bytes` for binary."
    m = await self.get(path, fields='content')
    return b64decode(m['content']) if m['format']=='base64' else m['content']

@patch
async def listing(self:JupyAsyncFilesClient, path=''):
    "The entries of directory `path`."
    return (await self.get(path))['content']


Let's start a local rustygate process with a temporary directory as its files root. All the following writes stay in that directory:

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('rustygate', '--root', root))
fc = JupyAsyncFilesClient(g.url)
m = await fc.write('notes.txt', 'hello')
m

```python
{ 'mtime': 1789266815.6835597,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

`write` returns the file model. Directory listings return those same stat fields without reading file contents:

In [ ]:
test_eq(await fc.read('notes.txt'), 'hello')
entry = next(e for e in await fc.listing() if e['name']=='notes.txt')
test_eq(entry, m)
entry

```python
{ 'mtime': 1789266815.6835597,
  'name': 'notes.txt',
  'path': 'notes.txt',
  'size': 5,
  'type': 'file',
  'writable': True}
```

Writing an existing file replaces its contents:

In [ ]:
await fc.write('notes.txt', 'hello2')
test_eq(await fc.read('notes.txt'), 'hello2')
await fc.read('notes.txt')

'hello2'

Pass `overwrite=False` to require a new file. An existing path raises `APIError` with status 409 and keeps its content:

In [ ]:
await fc.write('fresh.txt', 'made', overwrite=False)
with expect_fail(APIError): await fc.write('fresh.txt', 'again', overwrite=False)
test_eq(await fc.read('fresh.txt'), 'made')

`write` base64-encodes bytes for transport. `read` returns text when the stored bytes are valid UTF-8, otherwise bytes. Here the binary data round-trips unchanged:

In [ ]:
raw = bytes(range(256))
await fc.write('blob.bin', raw)
back = await fc.read('blob.bin')
test_eq(back, raw)
len(back)

256

`mkdir`, `rename`, and `copy` wrap the contents methods. Use `parents=True` to create missing directory ancestors.

A rename never overwrites an existing destination: it raises `APIError` with status 409. To replace a destination, delete it explicitly first:


In [ ]:
#| export
@patch
async def mkdir(self:JupyAsyncFilesClient, path, parents=False):
    "Create directory `path`; `parents` creates missing ancestors like `mkdir -p`."
    return await self.put(path, parents=parents or None, type='directory')

@patch
async def rename(self:JupyAsyncFilesClient, path, to):
    "Rename and return the new model. An existing destination raises APIError (409); it is never overwritten."
    return await self.patch(path, path=to)

@patch
async def copy(self:JupyAsyncFilesClient, src, to, unique=False, overwrite=True):
    "Copy and return the new model. `unique` chooses a free `name_n.ext`; `overwrite=False` requires a new file."
    return await self.post(to, overwrite=overwrite, unique=unique or None, copy_from=src)

In [ ]:
await fc.mkdir('sub')
await fc.mkdir('deep/a/b', parents=True)
await fc.copy('notes.txt', 'sub/notes.txt')
await fc.rename('sub/notes.txt', 'sub/renamed.txt')
with expect_fail(APIError): await fc.rename('sub/renamed.txt', 'notes.txt')
test_eq([e['name'] for e in await fc.listing('sub')], ['renamed.txt'])
await fc.delete('sub/renamed.txt')
await fc.delete('sub')
await fc.delete('blob.bin')
[e['name'] for e in await fc.listing()]

['deep', 'fresh.txt', 'notes.txt']

`search` returns `/api/search` results as `{'paths', 'complete'}`. It can search file names or content. With `up`, it searches ancestors for the nearest matching entry, such as `.git`, a project file, or a per-directory configuration file:

In [ ]:
#| export
@patch
async def search(self:JupyAsyncFilesClient, path='', **kwargs):
    "The `/api/search` result for directory `path`: `kwargs` are its query parameters (`q`, `up`, `glob`, ...), documented in rustygate's DEV.md"
    return await self._op(self.api.search.search, path=path or None, **kwargs)

In [ ]:
test_eq((await fc.search(path='deep/a/b', up='notes.txt'))['paths'], ['notes.txt'])
await fc.search(glob='*.txt')


```python
{'complete': True, 'paths': ['fresh.txt', 'notes.txt']}
```

Pass `unique=True` to write or copy under the first free `name_n.ext` instead of overwriting. The response contains the final name and path:

In [ ]:
m3 = await fc.write('notes.txt', 'variant', unique=True)
test_eq(m3['name'], 'notes_1.txt')
m4 = await fc.copy('notes.txt', 'notes.txt', unique=True)
test_eq(m4['name'], 'notes_2.txt')
test_eq(await fc.read('notes_2.txt'), await fc.read('notes.txt'))
m4['path']

'notes_2.txt'

Use `edit` to change a JSON file. It reads the content, calls `f` to modify the parsed object in place, and writes the result once. Like any file write, it replaces intervening changes:

In [ ]:
#| export
@patch
async def edit(self:JupyAsyncFilesClient, path, f):
    "Read the JSON file at `path`, apply `f` to the parsed object, and write it back."
    o = json.loads(await self.read(path))
    f(o)
    return await self.write(path, json.dumps(o, sort_keys=True, indent=1))

In [ ]:
await fc.write('cfg.json', json.dumps(dict(a=1)))
def _bump(o): o['a'] += 1
await fc.edit('cfg.json', _bump)
test_eq(json.loads(await fc.read('cfg.json'))['a'], 2)

## The cells client

Construct `JupyAsyncCellsClient` with a notebook path or `kernel_id`. Kernel-addressed cell operations follow the current binding through rename and restart. Inherited file methods still take explicit paths.

`view` returns the selected `cells` with their notebook `path`. Pass `fields='meta'` to include notebook `metadata`. `cells` returns the selected cell list.

`await client[id]` returns one cell; a tuple of ids returns a list.

In [ ]:
#| export
class JupyAsyncCellsClient(JupyAsyncFilesClient):
    "One notebook's cells over the gateway's cells API."
    def __init__(self, base_url, path=None, token=None, headers=None, timeout=30, http_client=None, verify=True, kernel_id=None):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        if (path is None)==(kernel_id is None): raise ValueError('Pass path or kernel_id')
        self.path,self.kernel_id = None if path is None else str(path),kernel_id

    def __getitem__(self, ids): return self._lookup(ids)

In [ ]:
#| export
def _cs(v):
    "A comma-separated str from a str, an int, an iterable, or None"
    if v is None or isinstance(v, str): return v
    return ','.join(map(str, v)) if hasattr(v, '__iter__') else str(v)

@patch
async def view(
    self:JupyAsyncCellsClient,
    ids=None, # Cell ids to keep: comma-separated str, or a list
    idx=None, # Cell positions to keep: comma-separated str, or a list of ints, 0-based, negative from the end; unions with `ids`
    section=None, # Cell id selecting a heading and its descendants; a non-heading selects itself
    ancestors=None, # Cell id whose enclosing headings to select, outermost first, excluding the cell itself
    before=None, # Cell id: select every cell before it, excluding it
    after=None, # Cell id: select every cell after it, excluding it
    q=None, # Keep only cells whose source matches this regex (multiline, smart-case)
    cell_type=None, # Keep only cells of this type: 'code', 'markdown', or 'raw'
    meta=None, # Keep only cells whose metadata contains this dict as a recursive subset; a None value means "key present"
    meta_not=None, # Drop cells whose metadata contains this dict as a recursive subset
    limit=None, # Keep at most this many cells after filtering
    context=None, # Also return this many neighbours either side of each kept cell; `matched` in the result names the true matches
    fields=None, # Comma-separated cell fields; '*' includes attachments, 'meta' adds notebook metadata; None omits attachments
):
    "The selected cells with their notebook path, requested metadata and matched ids."
    return await self._op(self.api.cells.get_cells if self.kernel_id is None else self.api.kernels.get_kernel_cells,
        **(dict(path=self.path) if self.kernel_id is None else dict(kid=self.kernel_id)),
        ids=_cs(ids), idx=_cs(idx), section=section, ancestors=ancestors, before=before, after=after, q=q, cell_type=cell_type,
        meta=None if meta is None else json.dumps(meta), meta_not=None if meta_not is None else json.dumps(meta_not),
        limit=limit, context=context, fields=fields)

@patch
@delegates(JupyAsyncCellsClient.view)
async def cells(self:JupyAsyncCellsClient, ids=None, **kwargs):
    "The selected cells in document order."
    return (await self.view(ids=ids, **kwargs))['cells']


`apply` submits operations in order and returns the added ids. `_lookup` implements the bracket notation:

In [ ]:
#| export
@patch
async def apply(self:JupyAsyncCellsClient, ops):
    "Apply `ops` in order, returning ids of added cells."
    m = await self._op(self.api.cells.post_cells if self.kernel_id is None else self.api.kernels.post_kernel_cells,
        **(dict(path=self.path) if self.kernel_id is None else dict(kid=self.kernel_id)), ops=ops)
    return m['added_ids']

@patch
async def _lookup(self:JupyAsyncCellsClient, ids):
    one = isinstance(ids, str)
    want = [ids] if one else list(ids)
    got = await self.cells(ids=want)
    if len(got)!=len(want): raise KeyError(', '.join(i for i in want if i not in {c['id'] for c in got}))
    return got[0] if one else got

We'll create a notebook through the files client. Any nbformat producer can write the file. For an unbound notebook, the cells API reads it from disk. For a notebook bound to a kernel, the gateway serves its held state:

In [ ]:
cells = [dict(id='aaa1', cell_type='code', source='1+1', metadata={}, outputs=[], execution_count=None),
    dict(id='bbb2', cell_type='markdown', source='# hi', metadata={})]
await fc.write('d.ipynb', json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=cells)))
nb = JupyAsyncCellsClient(g.url, 'd.ipynb')
[c['id'] for c in await nb.cells()]

['aaa1', 'bbb2']

Bracket lookup works like fastlite's id lookup, with `await` for the request. A missing id raises `KeyError`. Multiple cells come back in document order:

In [ ]:
c = await nb['bbb2']
test_eq(c['source'], '# hi')
pair = await nb['aaa1','bbb2']
test_eq([c['id'] for c in pair], ['aaa1','bbb2'])
try: await nb['nope']
except KeyError as e: err = str(e)
err

"'nope'"

The server fills in omitted defaults for a sparse `add` and returns added ids in operation order. An `update` replaces the fields you supply rather than merging their contents. Operations are applied in order, without rollback of earlier operations if a later one fails:

In [ ]:
added = await nb.apply([
    dict(op='add', cell=dict(cell_type='code', source='2+2'), after='aaa1'),
    dict(op='update', id='bbb2', source='# hello'),
])
new_id, = added
[c['id'] for c in await nb.cells()]

['aaa1', '97270c84', 'bbb2']

The cells API cannot edit an unbound file that doesn't parse as a notebook: it raises `APIError` with status 409. Deleting a kernel-bound path also returns 409: shut down the kernel before deleting its file.

In [ ]:
await fc.write('junk.ipynb', '{not json')
bad = JupyAsyncCellsClient(g.url, 'junk.ipynb')
try: await bad.apply([dict(op='update', id='aaa1', source='x')])
except APIError as e: code = e.status_code
test_eq(code, 409)

## Applying ops

`apply_ops` updates a list of cell dictionaries in place from broadcast operations. A client can keep its local view current without fetching the notebook after each change.

The gateway converts richer request operations, such as cell metadata `merge`, into add/update/delete broadcasts containing their results. Notebook-level `meta` leaves the cell list unchanged. Handle file-level `rename` and attachment events separately; passing them to this helper raises `ValueError`.

In [ ]:
#| export
def _update_cell(cell, fields):
    cell.update({k:v for k,v in fields.items() if k not in ('op','id')})
    if cell['cell_type']=='code':
        cell.setdefault('outputs', [])
        cell.setdefault('execution_count', None)
    else:
        cell.pop('outputs', None)
        cell.pop('execution_count', None)

In [ ]:
#| export
def apply_ops(cells, ops):
    "Apply cell broadcast operations in order to `cells` in place and return that list."
    for o in ops:
        ids = [c['id'] for c in cells]
        op = o['op']
        if op=='update' and o['id'] in ids: _update_cell(cells[ids.index(o['id'])], o)
        elif op in ('add','update'):
            c = dict(o['cell']) if op=='add' else {k:v for k,v in o.items() if k!='op'}
            if c.get('id') in ids: _update_cell(cells[ids.index(c['id'])], c)
            else:
                c.setdefault('cell_type', 'code')
                c.setdefault('metadata', {})
                if c['cell_type']=='code':
                    c.setdefault('outputs', [])
                    c.setdefault('execution_count', None)
                anchor = o.get('after') or o.get('before')
                at = ids.index(anchor) + bool(o.get('after')) if anchor in ids else len(cells)
                cells.insert(at, c)
        elif op=='delete':
            if o['id'] in ids: del cells[ids.index(o['id'])]
        elif op=='meta': pass
        else: raise ValueError(f"unhandled op: {op}")
    return cells

Here we add a code cell, delete another cell, and turn the new cell into a Markdown heading. The type change removes its execution count and outputs. `ancestors` selects the enclosing headings of a cell. `section` selects a heading and its descendants. `before` and `after` select the cells on one side of a cell. `fields='id'` returns just the IDs in cell dictionaries:

In [ ]:
view = await nb.cells()
ops = [dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3', metadata={}, outputs=[], execution_count=None), before='aaa1'),
    dict(op='delete', id='bbb2'), dict(op='update', id='ccc3', cell_type='markdown', source='# Calculation')]
await nb.apply(ops)
apply_ops(view, ops)
test_eq(view, await nb.cells())
test_eq(await nb.cells(ancestors='aaa1', fields='id'), [dict(id='ccc3')])
test_eq(await nb.cells(section='ccc3', cell_type='code', fields='id'), [dict(id='aaa1'), dict(id=new_id)])
test_eq(await nb.cells(before='aaa1', fields='id'), [dict(id='ccc3')])
test_eq(await nb.cells(after='aaa1', fields='id'), [dict(id=new_id)])
[(c['id'], c['cell_type']) for c in view]

[('ccc3', 'markdown'), ('aaa1', 'code'), ('97270c84', 'code')]

Cell operations tolerate missing and duplicate ids in these cases:

- Updating a missing id adds the cell at the end. Defaults include `cell_type='code'`, empty metadata, and, for code cells, empty outputs and a null execution count.
- Adding an existing id updates that cell in place and ignores the anchor.
- Adding relative to a missing anchor appends the cell.
- Deleting a missing id has no effect.

These rules do not make malformed operations valid. Here the duplicate add turns `ccc3` back into code. Its outputs start empty and its execution count is null. The local result matches the server:

In [ ]:
ops = [dict(op='update', id='zzz9', source='9'), dict(op='delete', id='zzz8'),
    dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3b'), after='ggg7'), dict(op='add', cell=dict(id='ddd4', source='4'), after='ggg7')]
await nb.apply(ops)
apply_ops(view, ops)
srv = await nb.cells()
test_eq([c['id'] for c in view], [c['id'] for c in srv])
for i in ('zzz9', 'ccc3', 'ddd4'): test_eq(next(c for c in view if c['id']==i), next(c for c in srv if c['id']==i))


## Change broadcasts

Create a kernel with `path` to subscribe its websocket clients to that notebook's changes. Broadcasts use message type `cell_ops` on channel `cells`, with content `path` and `ops`. Every subscriber receives accepted changes, including their author:

In [ ]:
kc = JupyAsyncKernelClient(g.url)
await kc.start_kernel(path='d.ipynb')
qs = JmsgQueues(kc, queues=('jmsg',), merge=dict(iopub='jmsg', stdin='jmsg', cells='jmsg'))
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running

True

In [ ]:
await nb.apply([dict(op='update', id='aaa1', source='40+2')])
m = await qs.jmsg_for('cell_ops', timeout=15)
test_eq(m['header']['msg_type'], 'cell_ops')
m['content']

{'path': 'd.ipynb',
 'ops': [{'op': 'update',
   'id': 'aaa1',
   'cell_type': 'code',
   'execution_count': None,
   'metadata': {},
   'outputs': [],
   'source': '40+2'}]}

Apply the broadcast's operations to the local cells:

In [ ]:
apply_ops(view, m['content']['ops'])
test_eq(next(c['source'] for c in view if c['id']=='aaa1'), '40+2')
m['content']['path']

'd.ipynb'

Edits from a second cells client arrive through the same websocket. The `meta` operation merges notebook metadata; `null` deletes a key. Its broadcast contains the resulting metadata. `view(fields='meta', limit=0)` reads notebook information without returning cells:

In [ ]:
own = JupyAsyncCellsClient(g.url, kernel_id=kc.kernel_id)
await own.apply([dict(op='update', id='aaa1', source='6*7'), dict(op='meta', metadata=dict(topic='arithmetic'))])
m = await qs.jmsg_for('cell_ops', timeout=5)
test_eq(next(c['source'] for c in apply_ops(view, m['content']['ops']) if c['id']=='aaa1'), '6*7')
info = await own.view(fields='meta', limit=0)
test_eq((info['path'], info['metadata']['topic'], info['cells']), ('d.ipynb', 'arithmetic', []))
info

To save execution output into the bound notebook, put the cell id in the request metadata as `cellId`. The gateway updates that cell's held state before forwarding its iopub messages. A cells fetch after the execution's `idle` can read the saved output.

An execution without `cellId`, or with an id absent from the notebook, still runs but does not persist its output:

In [ ]:
kc.execute('6*7', metadata=dict(cellId='aaa1'))
await qs.jmsg_for('status', pred=lambda m: m['content']['execution_state'] == 'idle', timeout=15)
c, = await nb.cells(ids=['aaa1'])
test_eq(c['outputs'][0]['data']['text/plain'], '42')
test_eq(c['execution_count'], 1)

The cells GET applies selection and filters in order. `ids`, `idx`, `section`, `ancestors`, `before` and `after` select a union of cells in document order, without duplicates. Positions start at zero; negative positions count from the end. With no selectors, it starts with all cells. Missing IDs select nothing. A section includes its heading and ends before the next heading of the same or higher level. A non-heading selects itself. Ancestors exclude the addressed cell. `before` and `after` select every cell on one side of the addressed cell, excluding it. Collapse flags do not affect either selector.

Headings are Markdown cells whose first nonblank, non-directive line matches `^#{1,6} \w`. Lines starting with `#|` are directives. Ordinary text on the first remaining line makes the cell a non-heading, regardless of later lines.

The selected cells must pass `q`, `cell_type`, `meta`, and `meta_not` when supplied. `q` is a multiline, smart-case regex over source text. `limit` keeps the first matching cells, then `context` adds neighbours. When context is requested, the `view` result's `matched` lists the matches before that expansion.

`fields='id,source'` returns just those cell fields. Omit `fields` for full cells without attachments, or pass `fields='*'` to include attachments. `metadata` names cell metadata; `meta` requests notebook metadata in the response, as in `fields='*,meta'` for a complete notebook read.

In [ ]:
test_eq([c['id'] for c in await nb.cells(idx=[0,-1])], ['ccc3', 'ddd4'])
test_eq([c['id'] for c in await nb.cells(ids='zzz9', idx=[0])], ['ccc3', 'zzz9'])
test_eq([c['id'] for c in await nb.cells(q=r'\*')], ['aaa1'])


For `meta`, provide a dictionary that the cell's metadata must contain as a recursive subset. `meta_not` excludes cells matching that subset. A `None` value means the key must exist, whatever its value.

A metadata `merge` uses `None` differently: it deletes that key. This example adds a tag, finds the tagged cell, then removes the tag:

In [ ]:
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=1))])
test_eq([c['id'] for c in await nb.cells(meta=dict(tag=None))], ['zzz9'])
await nb.apply([dict(op='merge', id='zzz9', metadata=dict(tag=None))])

[]

With `context=1`, the result includes up to one neighbour on either side of each match. `matched` identifies the cells that matched the query:

In [ ]:
got = await nb.view(q=r'\*', context=1)
test_eq((len(got['cells']), got['matched']), (3, ['aaa1']))
[c['id'] for c in got['cells']]

['ccc3', 'aaa1', '97270c84']

A `{'op':'rename','to':...}` broadcast reports the new notebook path. A kernel-addressed cells client keeps using the same kernel id.

Broadcast adds and updates contain the resulting cell fields. They need no follow-up read for those fields. Attachment bytes travel separately: attachment events identify the affected attachment, and clients fetch its bytes when needed. Notebook metadata changes arrive as `meta` operations containing the resulting metadata.

While a notebook is bound, the gateway keeps its current parsed state in memory. A changed disk mtime prompts a reload. Valid foreign writes replace the held state, repairing duplicate ids and broadcasting ordinary change operations. Missing or malformed files leave the open buffer unchanged, without forced restoration; pending changes save normally.

After reconnecting, reload the binding and notebook. No missed messages are replayed.


In [ ]:
#| hide
await kc.shutdown_kernel()
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()